# ETL — HR Employee Attrition
**Etapas:** Extração → Validação → Transformação → Enriquecimento → Exportação

## 0. Dependências

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Extração

In [ ]:
RAW_FILE   = "HR-Employee-Attrition.csv"  
OUTPUT_DIR = "output_pbi"                 
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_raw = pd.read_csv(RAW_FILE)

print(f"Shape: {df_raw.shape[0]:,} linhas  ×  {df_raw.shape[1]} colunas")
print()
df_raw.head(3)


In [ ]:
# Tipos e nulos 
info = pd.DataFrame({
    "dtype"       : df_raw.dtypes,
    "nulos"       : df_raw.isnull().sum(),
    "nulos_%"     : (df_raw.isnull().mean() * 100).round(2),
    "únicos"      : df_raw.nunique(),
    "amostra"     : df_raw.iloc[0],
})
print(info.to_string())


## 2. Validação de qualidade

In [ ]:
import sys

erros   = []
avisos  = []

COLUNAS_OBRIGATORIAS = [
    "EmployeeNumber", "Age", "Attrition", "Department", "JobRole",
    "MonthlyIncome", "OverTime", "YearsAtCompany",
]
faltando = [c for c in COLUNAS_OBRIGATORIAS if c not in df_raw.columns]
if faltando:
    erros.append(f"Colunas obrigatórias ausentes: {faltando}")

nulos = df_raw.isnull().sum()
cols_com_nulos = nulos[nulos > 0]
if not cols_com_nulos.empty:
    avisos.append(f"Colunas com nulos:\n{cols_com_nulos.to_string()}")

# EmployeeNumber duplicado 
dupl = df_raw["EmployeeNumber"].duplicated().sum()
if dupl > 0:
    erros.append(f"EmployeeNumber com {dupl} duplicata(s) — verifique a fonte.")

# Faixas esperadas 
checks_range = {
    "Age"               : (18, 65),
    "MonthlyIncome"     : (1000, 25000),
    "PercentSalaryHike" : (0, 100),
    "TotalWorkingYears" : (0, 50),
    "YearsAtCompany"    : (0, 50),
}
for col, (lo, hi) in checks_range.items():
    if col in df_raw.columns:
        fora = df_raw[(df_raw[col] < lo) | (df_raw[col] > hi)]
        if not fora.empty:
            avisos.append(f"{col}: {len(fora)} valor(es) fora de [{lo}, {hi}]")

CATEGORIAS_ESPERADAS = {
    "Attrition"     : {"Yes", "No"},
    "OverTime"      : {"Yes", "No"},
    "Gender"        : {"Male", "Female"},
    "MaritalStatus" : {"Single", "Married", "Divorced"},
    "BusinessTravel": {"Travel_Rarely", "Travel_Frequently", "Non-Travel"},
}
for col, esperado in CATEGORIAS_ESPERADAS.items():
    if col in df_raw.columns:
        inesperado = set(df_raw[col].dropna().unique()) - esperado
        if inesperado:
            avisos.append(f"{col}: categorias inesperadas → {inesperado}")

print("RELATÓRIO DE VALIDAÇÃO")
if erros:
    for e in erros:
        print(f"   ERRO  : {e}")
    sys.exit("Pipeline interrompido — corrija os erros acima.")
else:
    print("   Nenhum erro crítico encontrado.")

if avisos:
    for a in avisos:
        print(f"    AVISO : {a}")
else:
    print("   Nenhum aviso.")



## 3. Transformação

In [ ]:
df = df_raw.copy()

# Remover colunas constantes / sem utilidade analítica
COLUNAS_DESCARTAR = ["EmployeeCount", "Over18", "StandardHours"]
df.drop(columns=[c for c in COLUNAS_DESCARTAR if c in df.columns], inplace=True)
print(f" Colunas removidas: {COLUNAS_DESCARTAR}")


In [ ]:
#  Renomear colunas 
MAPA_COLUNAS = {
    "Age"                     : "idade",
    "Attrition"               : "attrition",
    "BusinessTravel"          : "frequencia_viagem",
    "DailyRate"               : "taxa_diaria",
    "Department"              : "departamento",
    "DistanceFromHome"        : "distancia_casa_km",
    "Education"               : "nivel_educacao_cod",
    "EducationField"          : "area_formacao",
    "EmployeeNumber"          : "id_funcionario",
    "EnvironmentSatisfaction" : "satisfacao_ambiente_cod",
    "Gender"                  : "genero",
    "HourlyRate"              : "taxa_horaria",
    "JobInvolvement"          : "envolvimento_trabalho_cod",
    "JobLevel"                : "nivel_cargo_cod",
    "JobRole"                 : "cargo",
    "JobSatisfaction"         : "satisfacao_trabalho_cod",
    "MaritalStatus"           : "estado_civil",
    "MonthlyIncome"           : "renda_mensal",
    "MonthlyRate"             : "taxa_mensal",
    "NumCompaniesWorked"      : "qtd_empresas_anteriores",
    "OverTime"                : "horas_extras",
    "PercentSalaryHike"       : "perc_aumento_salarial",
    "PerformanceRating"       : "avaliacao_desempenho_cod",
    "RelationshipSatisfaction": "satisfacao_relacionamento_cod",
    "StockOptionLevel"        : "nivel_stock_option",
    "TotalWorkingYears"       : "anos_experiencia_total",
    "TrainingTimesLastYear"   : "treinamentos_ultimo_ano",
    "WorkLifeBalance"         : "equilibrio_vida_trabalho_cod",
    "YearsAtCompany"          : "anos_empresa",
    "YearsInCurrentRole"      : "anos_cargo_atual",
    "YearsSinceLastPromotion" : "anos_desde_ultima_promocao",
    "YearsWithCurrManager"    : "anos_mesmo_gestor",
}
df.rename(columns=MAPA_COLUNAS, inplace=True)
print(f" Colunas renomeadas: {len(MAPA_COLUNAS)}")
df.columns.tolist()


In [ ]:
# Converter tipos

# Categóricas nominais 
CAT_NOMINAIS = [
    "frequencia_viagem", "departamento", "area_formacao",
    "genero", "cargo", "estado_civil",
]
for col in CAT_NOMINAIS:
    df[col] = pd.Categorical(df[col])

# Flags booleanas
FLAGS_BINARIAS = ["attrition", "horas_extras"]
for col in FLAGS_BINARIAS:
    df[col] = df[col].map({"Yes": True, "No": False}).astype("boolean")

print(" Tipos convertidos.")
df.dtypes


In [ ]:
# Decodificar escalas ordinais 

ESCALA_SATISFACAO = {1: "Baixo", 2: "Médio", 3: "Alto", 4: "Muito Alto"}
ESCALA_LIKERT_4   = {1: "Ruim",  2: "Regular", 3: "Bom", 4: "Excelente"}

MAPA_ORDINAIS = {
    "nivel_educacao_cod": {
        1: "Ensino Médio", 2: "Técnico", 3: "Graduação",
        4: "Mestrado",     5: "Doutorado",
    },
    "satisfacao_ambiente_cod"      : ESCALA_SATISFACAO,
    "envolvimento_trabalho_cod"    : ESCALA_SATISFACAO,
    "satisfacao_trabalho_cod"      : ESCALA_SATISFACAO,
    "satisfacao_relacionamento_cod": ESCALA_SATISFACAO,
    "equilibrio_vida_trabalho_cod" : {
        1: "Ruim", 2: "Regular", 3: "Bom", 4: "Ótimo"
    },
    "avaliacao_desempenho_cod"     : {3: "Excelente", 4: "Excepcional"},
    "nivel_cargo_cod"              : {
        1: "Júnior", 2: "Pleno", 3: "Sênior", 4: "Especialista", 5: "Diretor"
    },
}

ORDENS = {
    "nivel_educacao_cod"      : ["Ensino Médio","Técnico","Graduação","Mestrado","Doutorado"],
    "satisfacao_ambiente_cod" : ["Baixo","Médio","Alto","Muito Alto"],
    "envolvimento_trabalho_cod": ["Baixo","Médio","Alto","Muito Alto"],
    "satisfacao_trabalho_cod" : ["Baixo","Médio","Alto","Muito Alto"],
    "satisfacao_relacionamento_cod": ["Baixo","Médio","Alto","Muito Alto"],
    "equilibrio_vida_trabalho_cod" : ["Ruim","Regular","Bom","Ótimo"],
    "avaliacao_desempenho_cod"     : ["Excelente","Excepcional"],
    "nivel_cargo_cod"              : ["Júnior","Pleno","Sênior","Especialista","Diretor"],
}

for cod_col, mapa in MAPA_ORDINAIS.items():
    novo_col = cod_col.replace("_cod", "")
    ordem    = ORDENS.get(cod_col)
    df[novo_col] = pd.Categorical(
        df[cod_col].map(mapa),
        categories=ordem,
        ordered=True,
    )

print("Ordinais decodificadas.")
# Verificação 
colunas_novas = [c.replace("_cod","") for c in MAPA_ORDINAIS]
df[colunas_novas].head(3)


In [ ]:
# Padronizar textos categóricos

# rótulos em português - viagens
df["frequencia_viagem"] = df["frequencia_viagem"].map({
    "Non-Travel"        : "Sem Viagem",
    "Travel_Rarely"     : "Viaja Raramente",
    "Travel_Frequently" : "Viaja Frequentemente",
})

df["genero"] = df["genero"].map({"Male": "Masculino", "Female": "Feminino"})

df["estado_civil"] = df["estado_civil"].map({
    "Single"  : "Solteiro(a)",
    "Married" : "Casado(a)",
    "Divorced": "Divorciado(a)",
})

print("Textos padronizados.")


## 4. Enriquecimento

Criação de métricas derivadas e features prontas para uso nos visuais do Power BI.

In [ ]:
# Flags de risco 

# Risco de attrition — combinação de fatores observados
df["flag_horas_extras"]          = df["horas_extras"].astype(bool)
df["flag_sem_promocao_3a"]       = df["anos_desde_ultima_promocao"] >= 3
df["flag_distancia_alta"]        = df["distancia_casa_km"] > 15
df["flag_satisfacao_baixa"]      = df["satisfacao_trabalho_cod"] <= 2
df["flag_renda_abaixo_mediana"]  = df["renda_mensal"] < df["renda_mensal"].median()

# Score de risco composto (0–5): soma das flags
FATORES_RISCO = [
    "flag_horas_extras",
    "flag_sem_promocao_3a",
    "flag_distancia_alta",
    "flag_satisfacao_baixa",
    "flag_renda_abaixo_mediana",
]
df["score_risco"] = df[FATORES_RISCO].sum(axis=1).astype(int)

df["nivel_risco"] = pd.cut(
    df["score_risco"],
    bins=[-1, 1, 3, 5],
    labels=["Baixo", "Médio", "Alto"],
).astype("category")

print("Flags e score de risco criados.")
df[["id_funcionario","score_risco","nivel_risco"] + FATORES_RISCO].head(5)


In [ ]:
# Faixas etárias e de renda

df["faixa_etaria"] = pd.cut(
    df["idade"],
    bins=[17, 25, 35, 45, 55, 99],
    labels=["18–25", "26–35", "36–45", "46–55", "55+"],
    ordered=True,
)

QUARTIS_RENDA = df["renda_mensal"].quantile([0.25, 0.50, 0.75]).values
df["faixa_renda"] = pd.cut(
    df["renda_mensal"],
    bins=[0, QUARTIS_RENDA[0], QUARTIS_RENDA[1], QUARTIS_RENDA[2], 99999],
    labels=["Q1 — Baixa", "Q2 — Média-Baixa", "Q3 — Média-Alta", "Q4 — Alta"],
    ordered=True,
)

print("Faixas etária e de renda criadas.")
print(f"  Quartis de renda: {QUARTIS_RENDA.astype(int).tolist()}")


In [ ]:
#Métricas de tenure e estabilidade

# Proporção da carreira na empresa atual
df["perc_carreira_empresa"] = (
    df["anos_empresa"] / df["anos_experiencia_total"].replace(0, np.nan)
).clip(0, 1).round(4)

# Tempo médio por empresa anterior (excluindo empresa atual)
anos_fora = df["anos_experiencia_total"] - df["anos_empresa"]
df["media_anos_por_empresa_anterior"] = (
    (anos_fora / df["qtd_empresas_anteriores"].replace(0, np.nan))
    .clip(0)
    .round(2)
)

# Estagnação no cargo atual: anos no cargo / anos na empresa
df["indice_estagnacao_cargo"] = (
    df["anos_cargo_atual"] / df["anos_empresa"].replace(0, np.nan)
).clip(0, 1).round(4)

print("Métricas de tenure criadas.")
df[["id_funcionario","perc_carreira_empresa",
    "media_anos_por_empresa_anterior","indice_estagnacao_cargo"]].head(4)


In [ ]:
# Custo estimado de reposição 

FATOR_REPOSICAO = {
    "Júnior"      : 0.50,
    "Pleno"       : 0.75,
    "Sênior"      : 1.00,
    "Especialista": 1.50,
    "Diretor"     : 2.00,
}

df["custo_reposicao_est"] = (
    df["nivel_cargo"].astype(str)
    .map(FATOR_REPOSICAO)
    .fillna(0.75)
    * df["renda_mensal"] * 12
).round(0).astype(int)

print("Custo de reposição estimado.")
df.groupby("nivel_cargo")["custo_reposicao_est"].mean().map("${:,.0f}".format)


---
## 5. Verificação pós-transformação

In [ ]:
print(f"Shape final: {df.shape[0]:,} linhas  ×  {df.shape[1]} colunas")
print()
print("Tipos por grupo:")
print(df.dtypes.value_counts().to_string())
print()
print("Nulos por coluna (somente com nulos > 0):")
nulos_final = df.isnull().sum()
nulos_final = nulos_final[nulos_final > 0]
if nulos_final.empty:
    print("  Nenhum nulo encontrado ")
else:
    print(nulos_final.to_string())
print()
print("Distribuição attrition:")
print(df["attrition"].value_counts().to_string())
print()
print("Distribuição nível de risco:")
print(df["nivel_risco"].value_counts().sort_index().to_string())

In [ ]:
# Estatísticas descritivas das métricas numéricas principais
df[[
    "idade", "renda_mensal", "anos_empresa",
    "anos_experiencia_total", "score_risco", "custo_reposicao_est",
]].describe().T.round(2)

## 6. Exportação para Power BI

São geradas **três tabelas** seguindo um modelo estrela simplificado:

| Tabela | Tipo | Descrição |
|--------|------|-----------|
| `fato_funcionarios.csv` | Fato | Uma linha por funcionário — métricas e chaves |
| `dim_cargos.csv` | Dimensão | Atributos de cargo e departamento |
| `dim_satisfacao.csv` | Dimensão | Scores de satisfação e bem-estar por funcionário |


In [ ]:
# Tabela fato — métricas principais por funcionário

COLUNAS_FATO = [
    "id_funcionario",
    "idade", "faixa_etaria", "genero", "estado_civil",
    "departamento", "cargo", "nivel_cargo", "nivel_cargo_cod",
    "area_formacao", "nivel_educacao",
    "renda_mensal", "faixa_renda", "taxa_diaria", "taxa_horaria",
    "perc_aumento_salarial", "nivel_stock_option",
    "anos_empresa", "anos_cargo_atual", "anos_desde_ultima_promocao",
    "anos_mesmo_gestor", "anos_experiencia_total", "qtd_empresas_anteriores",
    "horas_extras", "frequencia_viagem", "distancia_casa_km",
    "treinamentos_ultimo_ano",
    "perc_carreira_empresa", "media_anos_por_empresa_anterior",
    "indice_estagnacao_cargo", "custo_reposicao_est",
    "score_risco", "nivel_risco",
    *FATORES_RISCO,
    "attrition",
]

fato = df[[c for c in COLUNAS_FATO if c in df.columns]].copy()

for col in fato.select_dtypes(include="boolean").columns:
    fato[col] = fato[col].astype("Int64")

fato.to_csv(f"{OUTPUT_DIR}/fato_funcionarios.csv", index=False, encoding="utf-8-sig")
print(f"fato_funcionarios.csv  → {fato.shape[0]:,} linhas × {fato.shape[1]} colunas")

In [ ]:
# Dimensão cargo 

dim_cargos = (
    df[["cargo", "departamento", "nivel_cargo", "nivel_cargo_cod"]]
    .drop_duplicates()
    .sort_values(["departamento", "nivel_cargo_cod"])
    .reset_index(drop=True)
)
dim_cargos.insert(0, "id_cargo", dim_cargos.index + 1)

dim_cargos.to_csv(f"{OUTPUT_DIR}/dim_cargos.csv", index=False, encoding="utf-8-sig")
print(f"dim_cargos.csv         → {dim_cargos.shape[0]:,} linhas × {dim_cargos.shape[1]} colunas")
dim_cargos

In [ ]:
# Dimensão satisfação 

COLUNAS_SAT = [
    "id_funcionario",
    "satisfacao_trabalho", "satisfacao_trabalho_cod",
    "satisfacao_ambiente", "satisfacao_ambiente_cod",
    "satisfacao_relacionamento", "satisfacao_relacionamento_cod",
    "envolvimento_trabalho", "envolvimento_trabalho_cod",
    "equilibrio_vida_trabalho", "equilibrio_vida_trabalho_cod",
    "avaliacao_desempenho", "avaliacao_desempenho_cod",
]

dim_satisfacao = df[[c for c in COLUNAS_SAT if c in df.columns]].copy()

# Score composto de bem-estar (média dos 5 índices de satisfação, escala 1–4)
indices_numericos = [
    "satisfacao_trabalho_cod", "satisfacao_ambiente_cod",
    "satisfacao_relacionamento_cod", "envolvimento_trabalho_cod",
    "equilibrio_vida_trabalho_cod",
]
dim_satisfacao["score_bemestar"] = (
    dim_satisfacao[indices_numericos].mean(axis=1).round(2)
)

dim_satisfacao.to_csv(f"{OUTPUT_DIR}/dim_satisfacao.csv", index=False, encoding="utf-8-sig")
print(f"dim_satisfacao.csv     → {dim_satisfacao.shape[0]:,} linhas × {dim_satisfacao.shape[1]} colunas")
dim_satisfacao.head(4)


In [ ]:
# carga

from datetime import datetime

resumo = pd.DataFrame([
    {"tabela": "fato_funcionarios", "linhas": len(fato),
     "colunas": fato.shape[1], "arquivo": "fato_funcionarios.csv"},
    {"tabela": "dim_cargos",        "linhas": len(dim_cargos),
     "colunas": dim_cargos.shape[1], "arquivo": "dim_cargos.csv"},
    {"tabela": "dim_satisfacao",    "linhas": len(dim_satisfacao),
     "colunas": dim_satisfacao.shape[1], "arquivo": "dim_satisfacao.csv"},
])
resumo["gerado_em"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

resumo.to_csv(f"{OUTPUT_DIR}/_resumo_carga.csv", index=False, encoding="utf-8-sig")

print("ETL CONCLUÍDO")
print(f"Arquivos gerados em: ./{OUTPUT_DIR}/")
print()
print(resumo[["tabela","linhas","colunas","arquivo"]].to_string(index=False))